In [52]:
!pip install openai

In [1]:
import os
import openai

# 올바른 환경 변수 이름으로 설정
os.environ["OPENAI_API_KEY"] = "~~~~~~~~~~~~~"

# 또는 직접 openai.api_key 속성 설정
openai.api_key = "~~~~~~~~~~~~~"


In [54]:
from openai import OpenAI
import json

In [55]:
client = OpenAI()

In [56]:
# 데이터베이스에 직접 접근 하는 함수 or Spring의 API를 호출하는 함수를 정의
# dict 리턴 : json 형식을 조금 더 편하게 사용

# 독립적으로 작동할 수 있는(LLM을 거쳐서 호출하는거 말고) 함수를 생성
def search_hanger(height: str, width:str, depth: str) -> dict:

  # # ex) api 호출의 경우
  # import requests
  # response = requests.get(f"localhost:8080?height={height}&width={width}&depth={depth}")

  # # ex) db 접근의 경우
  # connection_str = "mysql:~~~~~~/db"

  # 실제 쿼리 접근 코드 또는 api 호출 코드를 넣어주세요.
  print("데이터베이스 쿼리 및 api 활용을 수행합니다.")

  search_result = {
    "hangerRacks": [
      {
        "id": "HR001",
        "description": "5-tier hanger rack",
        "dimensions": {
          "width": 800,
          "depth": 600,
          "height": 2100
        },
        "tiers": 5,
        "quantity": 2,
        "material": "Steel",
        "color": "Black",
        "loadCapacityPerTier": 50,
        "totalLoadCapacity": 250,
        "features": [
          "Adjustable shelves",
          "Anti-rust coating",
          "Non-slip feet"
        ]
      },
      {
        "id": "HR002",
        "description": "3-tier double hanger rack",
        "dimensions": {
          "width": 800,
          "depth": 600,
          "height": 2100
        },
        "tiers": 3,
        "quantity": 2,
        "material": "Steel",
        "color": "White",
        "loadCapacityPerTier": 40,
        "totalLoadCapacity": 240,
        "features": [
          "Double hanging rods",
          "Wheels for mobility",
          "Adjustable height"
        ]
      }
    ]
  }

  return search_result

search_hanger("a", "b", "c")

데이터베이스 쿼리 및 api 활용을 수행합니다.


{'hangerRacks': [{'id': 'HR001',
   'description': '5-tier hanger rack',
   'dimensions': {'width': 800, 'depth': 600, 'height': 2100},
   'tiers': 5,
   'quantity': 2,
   'material': 'Steel',
   'color': 'Black',
   'loadCapacityPerTier': 50,
   'totalLoadCapacity': 250,
   'features': ['Adjustable shelves', 'Anti-rust coating', 'Non-slip feet']},
  {'id': 'HR002',
   'description': '3-tier double hanger rack',
   'dimensions': {'width': 800, 'depth': 600, 'height': 2100},
   'tiers': 3,
   'quantity': 2,
   'material': 'Steel',
   'color': 'White',
   'loadCapacityPerTier': 40,
   'totalLoadCapacity': 240,
   'features': ['Double hanging rods',
    'Wheels for mobility',
    'Adjustable height']}]}

In [57]:
# llm이 search_hanger 함수를 알아낼 수 있게 도구를 정의
llm_search_tool = {
    "type": "function",
    "function":{
        "name": "search_hanger",
        "description": "고객이 찾고 있는 행거에 대한 검색 결과를 반환합니다. 고객이 행거에 대한 높이(height), 너비(width), 깊이(depth)를 정확히 입력한 경우에 이 함수를 호출하세요.",

        "parameters": {
            "type": "object",
            "properties": {
                "height": {
                    "type": "string",
                    "description": "행거의 높이"
                },
                "width": {
                    "type": "string",
                    "description": "행거의 너비(길이)"
                },
                "depth": {
                    "type": "string",
                    "description": "행거의 깊이"
                }
            },
            "required": ["height", "width", "depth"],
            "additionalProperties": False
        }
    }
}

tools = [ llm_search_tool ]

In [58]:
system_prompt = """

당신은 검색된 문서부터 질문의 답변을 작성하는 언어 모델입니다. 도구를 이용해 사용자를 지원합니다.

### 지시사항
당신은 사용자로부터 선반 랙의 높이, 너비(길이), 깊이를 확정받아서 검색을 수행하는 챗봇입니다.

1. 사용자가 행거의 높이(세로), 길이(가로), 깊이 이 세 가지 값을 모두 확정할 때까지 사용자에게 반문하세요. 이는 가장 중요합니다. 모든 것이 결정될 때까지 반문하십시오.
2. 단수 추가를 요청하는 경우 1세트 설치 시 2~3cm 여유공간이 필요하다고 안내하세요.
3. 모든 것이 확정되면 도구를 사용합니다.

예시)

User: 길이 3000, 높이 2100의 선반 랙을 사려고합니다.
Assistant: 네, 설치하고자 하는 선반 랙의 길이가 3000, 높이 2100이 맞으실까요? 깊이에 대한 정보도 알려주시면 선반을 검색해드리겠습니다.
User: 실제 공간 총 높이는 2300인데 조금 비워두려고 2100으로 하려합니다. 그리고 향후에 단수 추가되는 부분도 추가로 구성이 가능할까요?
Assistant: 1세트 설치 시 2~3cm 여유공간이 필요하신 점 참고 부탁드립니다.깊이는 500 정도로 괜찮으실까요?
User: 깊이는 700이 좋겠습니다.
Assistant: 네, 선반을 추천해드릴게요.
검색 결과:
"""

In [59]:
user_prompt = ""

In [60]:
response = client.chat.completions.create(
  model="gpt-4o",
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
  ],
  temperature=0,
  tools=tools
)

print(response.choices[0].message.content)

안녕하세요! 선반 랙을 찾고 계신가요? 설치하고자 하는 선반 랙의 높이, 너비(길이), 깊이를 알려주시면 검색을 도와드리겠습니다. 세 가지 정보를 모두 제공해 주시면 감사하겠습니다.


In [61]:
response = client.chat.completions.create(
  model="gpt-4o",
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "assistant", "content": "안녕하세요! 선반 랙을 찾고 계신가요? 설치하고자 하는 선반 랙의 높이, 너비(길이), 깊이를 알려주시면 검색을 도와드리겠습니다. 세 가지 정보를 모두 제공해 주시면 감사하겠습니다."},
    {"role": "user", "content": "아 선반 랙은 분홍색으로 찾고 있는데 높이는 1500 정도가 좋겠어요."} # 현재 질문
  ],
  temperature=0,
  tools=tools
)

print(response.choices[0].message.content)

네, 선반 랙의 높이가 1500으로 설정하셨군요. 너비(길이)와 깊이에 대한 정보도 알려주시면 검색을 도와드리겠습니다.


In [62]:
response = client.chat.completions.create(
  model="gpt-4o",
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "assistant", "content": "안녕하세요! 선반 랙을 찾고 계신가요? 설치하고자 하는 선반 랙의 높이, 너비(길이), 깊이를 알려주시면 검색을 도와드리겠습니다. 세 가지 정보를 모두 제공해 주시면 감사하겠습니다."},
    {"role": "user", "content": "아 선반 랙은 분홍색으로 찾고 있는데 높이는 1500 정도가 좋겠어요."},
    {"role": "assistant", "content": "네, 선반 랙의 높이가 1500으로 설정하셨군요. 이제 너비(길이)와 깊이에 대한 정보를 알려주시면 검색을 도와드리겠습니다."},
    {"role": "user", "content": "음 고민해볼게요. 혹시 향후에 단수 추가되는 부분도 추가로 구성이 가능할까요?"}, # 현재 질문
  ],
  temperature=0,
  tools=tools
)

print(response.choices[0].message.content)

네, 선반 랙을 설치할 때 1세트 설치 시 2~3cm 여유공간이 필요하다는 점을 참고하시면 좋습니다. 나중에 단수를 추가할 계획이 있으시다면 이 점을 고려해 주세요. 너비(길이)와 깊이에 대한 정보도 알려주시면 검색을 도와드리겠습니다.


In [63]:
response = client.chat.completions.create(
  model="gpt-4o",
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "assistant", "content": "안녕하세요! 선반 랙을 찾고 계신가요? 설치하고자 하는 선반 랙의 높이, 너비(길이), 깊이를 알려주시면 검색을 도와드리겠습니다. 세 가지 정보를 모두 제공해 주시면 감사하겠습니다."},
    {"role": "user", "content": "아 선반 랙은 분홍색으로 찾고 있는데 높이는 1500 정도가 좋겠어요."},
    {"role": "assistant", "content": "네, 선반 랙의 높이가 1500으로 설정하셨군요. 이제 너비(길이)와 깊이에 대한 정보를 알려주시면 검색을 도와드리겠습니다."},
    {"role": "user", "content": "음 고민해볼게요. 혹시 향후에 단수 추가되는 부분도 추가로 구성이 가능할까요?"},
    {"role": "assistant", "content": "1세트 설치 시 2~3cm 여유공간이 필요하신 점 참고 부탁드립니다. 단수 추가를 고려하신다면 이 점을 염두에 두시면 좋겠습니다. 너비(길이)와 깊이에 대한 정보도 알려주시면 검색을 도와드리겠습니다."},
    {"role": "user", "content": "아 정정할게요. 높이는 1200, 길이는 2000으로요."}, # 현재 질문
  ],
  temperature=0,
  tools=tools
)

print(response.choices[0].message.content)

네, 선반 랙의 높이가 1200, 길이가 2000으로 설정하셨군요. 이제 깊이에 대한 정보를 알려주시면 검색을 도와드리겠습니다.


In [64]:
response = client.chat.completions.create(
  model="gpt-4o",
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "assistant", "content": "안녕하세요! 선반 랙을 찾고 계신가요? 설치하고자 하는 선반 랙의 높이, 너비(길이), 깊이를 알려주시면 검색을 도와드리겠습니다. 세 가지 정보를 모두 제공해 주시면 감사하겠습니다."},
    {"role": "user", "content": "아 선반 랙은 분홍색으로 찾고 있는데 높이는 1500 정도가 좋겠어요."},
    {"role": "assistant", "content": "네, 선반 랙의 높이가 1500으로 설정하셨군요. 이제 너비(길이)와 깊이에 대한 정보를 알려주시면 검색을 도와드리겠습니다."},
    {"role": "user", "content": "음 고민해볼게요. 혹시 향후에 단수 추가되는 부분도 추가로 구성이 가능할까요?"},
    {"role": "assistant", "content": "1세트 설치 시 2~3cm 여유공간이 필요하신 점 참고 부탁드립니다. 단수 추가를 고려하신다면 이 점을 염두에 두시면 좋겠습니다. 너비(길이)와 깊이에 대한 정보도 알려주시면 검색을 도와드리겠습니다."},
    {"role": "user", "content": "아 정정할게요. 높이는 1200, 길이는 2000으로요."},
    {"role": "assistant", "content": "네, 선반 랙의 높이가 1200, 길이가 2000으로 설정하셨군요. 이제 깊이에 대한 정보를 알려주시면 검색을 도와드리겠습니다."},
    {"role": "user", "content": "깊이 추천 좀 해주세요."} # 현재 질문
  ],
  temperature=0,
  tools=tools
)

print(response.choices[0].message.content)

일반적으로 선반 랙의 깊이는 300mm에서 600mm 사이가 많이 사용됩니다. 공간의 용도와 수납할 물건의 크기에 따라 다르겠지만, 400mm에서 500mm 정도가 적당할 수 있습니다. 이 범위 내에서 괜찮으신가요, 아니면 다른 깊이를 원하시나요?


In [65]:
response = client.chat.completions.create(
  model="gpt-4o",
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "assistant", "content": "안녕하세요! 선반 랙을 찾고 계신가요? 설치하고자 하는 선반 랙의 높이, 너비(길이), 깊이를 알려주시면 검색을 도와드리겠습니다. 세 가지 정보를 모두 제공해 주시면 감사하겠습니다."},
    {"role": "user", "content": "아 선반 랙은 분홍색으로 찾고 있는데 높이는 1500 정도가 좋겠어요."},
    {"role": "assistant", "content": "네, 선반 랙의 높이가 1500으로 설정하셨군요. 이제 너비(길이)와 깊이에 대한 정보를 알려주시면 검색을 도와드리겠습니다."},
    {"role": "user", "content": "음 고민해볼게요. 혹시 향후에 단수 추가되는 부분도 추가로 구성이 가능할까요?"},
    {"role": "assistant", "content": "1세트 설치 시 2~3cm 여유공간이 필요하신 점 참고 부탁드립니다. 단수 추가를 고려하신다면 이 점을 염두에 두시면 좋겠습니다. 너비(길이)와 깊이에 대한 정보도 알려주시면 검색을 도와드리겠습니다."},
    {"role": "user", "content": "아 정정할게요. 높이는 1200, 길이는 2000으로요."},
    {"role": "assistant", "content": "네, 선반 랙의 높이가 1200, 길이가 2000으로 설정하셨군요. 이제 깊이에 대한 정보를 알려주시면 검색을 도와드리겠습니다."},
    {"role": "user", "content": "깊이 추천 좀 해주세요."},
    {"role": "assistant", "content": "일반적으로 선반 랙의 깊이는 300mm에서 600mm 사이가 많이 사용됩니다. 공간의 용도와 수납할 물품에 따라 다르겠지만, 500mm 정도가 적당할 수 있습니다. 이 깊이로 진행해도 괜찮으신가요?"},
    {"role": "user", "content": "사무실에서 쓸 선반이에요"} # 현재 질문
  ],
  temperature=0,
  tools=tools
)

print(response.choices[0].message.content)

사무실에서 사용하실 경우, 서류나 사무용품을 수납할 수 있는 깊이로 400mm에서 500mm 정도가 적당할 수 있습니다. 500mm 깊이로 진행해도 괜찮으신가요?


In [66]:
response = client.chat.completions.create(
  model="gpt-4o",
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "assistant", "content": "안녕하세요! 선반 랙을 찾고 계신가요? 설치하고자 하는 선반 랙의 높이, 너비(길이), 깊이를 알려주시면 검색을 도와드리겠습니다. 세 가지 정보를 모두 제공해 주시면 감사하겠습니다."},
    {"role": "user", "content": "아 선반 랙은 분홍색으로 찾고 있는데 높이는 1500 정도가 좋겠어요."},
    {"role": "assistant", "content": "네, 선반 랙의 높이가 1500으로 설정하셨군요. 이제 너비(길이)와 깊이에 대한 정보를 알려주시면 검색을 도와드리겠습니다."},
    {"role": "user", "content": "음 고민해볼게요. 혹시 향후에 단수 추가되는 부분도 추가로 구성이 가능할까요?"},
    {"role": "assistant", "content": "1세트 설치 시 2~3cm 여유공간이 필요하신 점 참고 부탁드립니다. 단수 추가를 고려하신다면 이 점을 염두에 두시면 좋겠습니다. 너비(길이)와 깊이에 대한 정보도 알려주시면 검색을 도와드리겠습니다."},
    {"role": "user", "content": "아 정정할게요. 높이는 1200, 길이는 2000으로요."},
    {"role": "assistant", "content": "네, 선반 랙의 높이가 1200, 길이가 2000으로 설정하셨군요. 이제 깊이에 대한 정보를 알려주시면 검색을 도와드리겠습니다."},
    {"role": "user", "content": "깊이 추천 좀 해주세요."},
    {"role": "assistant", "content": "일반적으로 선반 랙의 깊이는 사용 용도에 따라 다르지만, 보통 400mm에서 600mm 사이가 많이 사용됩니다. 만약 더 많은 수납 공간이 필요하시다면 700mm 이상의 깊이도 고려해보실 수 있습니다. 어떤 깊이가 좋으신가요?"},
    {"role": "user", "content": "사무실에서 쓸 선반이에요"},
    {"role": "assistant", "content": "사무실에서 사용하실 경우, 일반적으로 400mm에서 600mm 깊이의 선반이 적당합니다. 이 범위 내에서 선택하시면 좋을 것 같습니다. 어떤 깊이로 하시겠어요?"},
    {"role": "user", "content": "500 정도가 좋아보여요"},
  ],
  temperature=0,
  tools=tools
)

print(response.choices[0].message.content)

if response.choices[0].finish_reason == "tool_calls":
  arguments = json.loads(response.choices[0].message.tool_calls[0].function.arguments)
  search_result = search_hanger(**arguments)

좋습니다! 선반 랙의 높이는 1200, 너비(길이)는 2000, 깊이는 500으로 설정하셨습니다. 이제 이 정보를 바탕으로 선반을 검색해드리겠습니다. 잠시만 기다려 주세요.
데이터베이스 쿼리 및 api 활용을 수행합니다.


In [67]:
search_result

{'hangerRacks': [{'id': 'HR001',
   'description': '5-tier hanger rack',
   'dimensions': {'width': 800, 'depth': 600, 'height': 2100},
   'tiers': 5,
   'quantity': 2,
   'material': 'Steel',
   'color': 'Black',
   'loadCapacityPerTier': 50,
   'totalLoadCapacity': 250,
   'features': ['Adjustable shelves', 'Anti-rust coating', 'Non-slip feet']},
  {'id': 'HR002',
   'description': '3-tier double hanger rack',
   'dimensions': {'width': 800, 'depth': 600, 'height': 2100},
   'tiers': 3,
   'quantity': 2,
   'material': 'Steel',
   'color': 'White',
   'loadCapacityPerTier': 40,
   'totalLoadCapacity': 240,
   'features': ['Double hanging rods',
    'Wheels for mobility',
    'Adjustable height']}]}

In [68]:
response = client.chat.completions.create(
  model="gpt-4o",
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "assistant", "content": "안녕하세요! 선반 랙을 찾고 계신가요? 설치하고자 하는 선반 랙의 높이, 너비(길이), 깊이를 알려주시면 검색을 도와드리겠습니다. 세 가지 정보를 모두 제공해 주시면 감사하겠습니다."},
    {"role": "user", "content": "아 선반 랙은 분홍색으로 찾고 있는데 높이는 1500 정도가 좋겠어요."},
    {"role": "assistant", "content": "네, 선반 랙의 높이가 1500으로 설정하셨군요. 이제 너비(길이)와 깊이에 대한 정보를 알려주시면 검색을 도와드리겠습니다."},
    {"role": "user", "content": "음 고민해볼게요. 혹시 향후에 단수 추가되는 부분도 추가로 구성이 가능할까요?"},
    {"role": "assistant", "content": "1세트 설치 시 2~3cm 여유공간이 필요하신 점 참고 부탁드립니다. 단수 추가를 고려하신다면 이 점을 염두에 두시면 좋겠습니다. 너비(길이)와 깊이에 대한 정보도 알려주시면 검색을 도와드리겠습니다."},
    {"role": "user", "content": "아 정정할게요. 높이는 1200, 길이는 2000으로요."},
    {"role": "assistant", "content": "네, 선반 랙의 높이가 1200, 길이가 2000으로 설정하셨군요. 이제 깊이에 대한 정보를 알려주시면 검색을 도와드리겠습니다."},
    {"role": "user", "content": "깊이 추천 좀 해주세요."},
    {"role": "assistant", "content": "일반적으로 선반 랙의 깊이는 사용 용도에 따라 다르지만, 보통 400mm에서 600mm 사이가 많이 사용됩니다. 만약 더 많은 수납 공간이 필요하시다면 700mm 이상의 깊이도 고려해보실 수 있습니다. 어떤 깊이가 좋으신가요?"},
    {"role": "user", "content": "사무실에서 쓸 선반이에요"},
    {"role": "assistant", "content": "사무실에서 사용하실 경우, 일반적으로 400mm에서 600mm 깊이의 선반이 적당합니다. 이 범위 내에서 선택하시면 좋을 것 같습니다. 어떤 깊이로 하시겠어요?"},
    {"role": "user", "content": "500 정도가 좋아보여요"},
    {"role": "assistant", "content": "좋습니다! 선반 랙의 높이는 1200, 너비(길이)는 2000, 깊이는 500으로 설정하셨습니다. 이제 이 정보를 바탕으로 선반을 검색해드리겠습니다. 잠시만 기다려 주세요. 데이터베이스 쿼리 및 api 활용을 수행합니다."},

    # 아래 내용은 굳이 사용자에게 노출할 필요 없이(UI 없이) 사용하면 됩니다.(프로그래밍 적인 처리 필요)
    {"role": "user", "content": f"검색 결과: {search_result} 자 이제 검색 결과를 바탕으로 사용자에게 전형적인 LLM 답변을 작성하세요."},
  ],
  temperature=0,
  tools=tools
)
print(response)
print(response.choices[0].message.content)

ChatCompletion(id='chatcmpl-AImlI4s0dllfBVNaMSlr1IrQTwAeZ', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_wfL3LNXmyxgsY05CcotWnKzc', function=Function(arguments='{"height":"1200","width":"2000","depth":"500"}', name='search_hanger'), type='function')]))], created=1729040708, model='gpt-4o-2024-08-06', object='chat.completion', service_tier=None, system_fingerprint='fp_6b68a8204b', usage=CompletionUsage(completion_tokens=25, prompt_tokens=1235, total_tokens=1260, completion_tokens_details=CompletionTokensDetails(audio_tokens=None, reasoning_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=1152)))
None


In [69]:
system_prompt = """

당신은 검색된 문서부터 질문의 답변을 작성하는 언어 모델입니다. 도구를 이용해 사용자를 지원합니다.

### 지시사항
당신은 사용자로부터 선반 랙의 높이, 너비(길이), 깊이를 확정받아서 검색을 수행하는 챗봇입니다.

1. 사용자가 행거의 높이(세로), 길이(가로), 깊이 이 세 가지 값을 모두 확정할 때까지 사용자에게 반문하세요. 이는 가장 중요합니다. 모든 것이 결정될 때까지 반문하십시오.
2. 단수 추가를 요청하는 경우 1세트 설치 시 2~3cm 여유공간이 필요하다고 안내하세요.
3. 모든 것이 확정되면 도구를 사용합니다.

예시)

User: 길이 3000, 높이 2100의 선반 랙을 사려고합니다.
Assistant: 네, 설치하고자 하는 선반 랙의 길이가 3000, 높이 2100이 맞으실까요? 깊이에 대한 정보도 알려주시면 선반을 검색해드리겠습니다.
User: 실제 공간 총 높이는 2300인데 조금 비워두려고 2100으로 하려합니다. 그리고 향후에 단수 추가되는 부분도 추가로 구성이 가능할까요?
Assistant: 1세트 설치 시 2~3cm 여유공간이 필요하신 점 참고 부탁드립니다.깊이는 500 정도로 괜찮으실까요?
User: 깊이는 700이 좋겠습니다.
Assistant: 네, 선반을 추천해드릴게요.
검색 결과:
"""

user_prompt = """"""

messages = [
  {"role": "system", "content": system_prompt},
]

def chatbot_search(user_message):
  global messages

  messages.append({"role": "user", "content": user_message})

  response = client.chat.completions.create(
    model="gpt-4o",
    messages=messages,
    tools=tools
  )
  chatbot_message = response.choices[0].message.content
  messages.append({"role": "assistant", "content": chatbot_message})

  if response.choices[0].finish_reason == "tool_calls":

    arguments = json.loads(response.choices[0].message.tool_calls[0].function.arguments)
    search_result = search_hanger(**arguments)

    messages.append({"role": "user", "content": f"검색 결과: {search_result} 자 이제 검색 결과를 바탕으로 사용자에게 전형적인 LLM 답변을 작성하세요."})
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=tools
    )
    # print("결과", response)
    chatbot_message = response.choices[0].message.content
    messages = [ {"role": "system", "content": system_prompt}]
  return chatbot_message

In [70]:
# 사용자가 채팅창에 접속
chatbot_search("")

'안녕하세요! 선반 랙의 높이, 너비(길이), 깊이를 알려주시면 검색을 도와드리겠습니다. 해당 정보들을 모두 제공해 주세요.'

In [71]:
print(chatbot_search("네, 선반을 찾고 있습니다. 사무실에서 사용할거에요"))

알겠습니다. 선반을 설치하려는 공간의 높이, 너비(길이), 깊이 세 가지 정보를 정확히 알려주시면 검색을 도와드리겠습니다. 각 값을 알려주시면 감사하겠습니다.


In [72]:
print(chatbot_search("크기 추천좀 해줄래?"))

설치할 공간과 용도에 따라 다양하게 선택할 수 있습니다. 기본적으로 공간에 맞는 적절한 높이, 너비(길이), 깊이를 고려해야 하는데요. 우선 사용할 공간의 높이, 너비(길이), 깊이를 말씀해 주시면, 어떤 크기가 적당한지 도와드릴 수 있어요!


In [73]:
print(chatbot_search("높이는 2000 정도가 괜찮을것 같아. 너비와 깊이를 용도에 맞게 설명해줘"))

좋습니다! 높이를 2000으로 고려하고 계시군요.

- **너비(길이)**: 선반의 너비는 어떤 물건들을 올릴지에 따라 달라질 수 있습니다. 일반적인 사무실 용도로는 A4 파일, 서적, 사무기기 등을 수납할 수 있는 800mm에서 1200mm 정도를 추천드립니다. 더 많은 물건을 수납해야 한다면 너비를 늘리는 것도 고려해보세요.

- **깊이**: 깊이는 선반에 올릴 물건의 크기에 따라 달라집니다. 일반적인 사무용 선반의 경우, 300mm에서 600mm 사이의 깊이가 흔하게 사용됩니다. 500mm 정도면 다양한 물품을 안정적으로 수납할 수 있을 것입니다.

이제 너비(길이)와 깊이에 대해 선택하신 뒤 알려주시면, 검색을 진행하도록 하겠습니다!


In [74]:
print(chatbot_search("가로 길이가 800이면 좀 짧은가?"))

너비(길이) 800mm는 일반 사무실 용도로 적당할 수 있지만, 수납하려는 물품의 크기와 양에 따라 결정할 필요가 있습니다. 여러 물건을 한 번에 수납해야 할 경우 좁다고 느낄 수도 있습니다.

선반의 총 공간과 수납할 물품의 크기를 고려하여 800mm로 충분할지 판단하시는 게 좋습니다. 만약 추가로 확장할 가능성을 염두에 두신다면, 1000mm 이상을 고려하셔도 좋습니다.

현재 너비를 800mm로 정하고 싶으신가요? 그리고 깊이는 얼마나 원하시는지 말씀해 주세요!


In [75]:
print(chatbot_search("깊이를 500으로 설정해줘"))

데이터베이스 쿼리 및 api 활용을 수행합니다.
검색 결과 두 가지의 선반 랙 옵션이 있습니다. 아래 내용을 참고하여 필요에 맞게 선택해 보세요:

1. **5-tier Hanger Rack (ID: HR001)**
   - **설명**: 5단 행거 랙
   - **크기**: 너비 800mm, 깊이 600mm, 높이 2100mm
   - **특징**:
     - 조절 가능한 선반
     - 녹 방지 코팅
     - 미끄럼 방지 발
   - **재질 및 색상**: 철재, 검정색
   - **적재 용량**: 선반당 50kg, 총 250kg
   - **구성 수량**: 2개

2. **3-tier Double Hanger Rack (ID: HR002)**
   - **설명**: 3단 더블 행거 랙
   - **크기**: 너비 800mm, 깊이 600mm, 높이 2100mm
   - **특징**:
     - 더블 행거봉
     - 이동용 바퀴
     - 높이 조절 가능
   - **재질 및 색상**: 철재, 흰색
   - **적재 용량**: 선반당 40kg, 총 240kg
   - **구성 수량**: 2개

두 제품 모두 사무실 용도로 적합하며, 공간에 따라 가용성을 확인해 보세요. 추가로 1세트 설치 시 2~3cm 여유 공간을 두는 점도 참고 부탁드립니다. 추가 정보가 필요하시면 언제든지 말씀해 주세요!
